In [3]:
import pandas as pd
import random
from datetime import datetime, timedelta
import numpy as np

# Veri üretimi için ayarlar
num_rows = 1000
random.seed(42) # Her çalıştırmada aynı veriyi üretmesi için
np.random.seed(42)

# E-Ticaret Veri Havuzları
kategoriler = ['Elektronik', 'Giyim', 'Kozmetik', 'Ev Aletleri', 'Spor', 'Kitap']
sehirler = ['İstanbul', 'Ankara', 'İzmir', 'Bursa', 'Antalya', 'Adana']
durumlar = ['Tamamlandı', 'Tamamlandı', 'Tamamlandı', 'Beklemede', 'İptal Edildi'] # Tamamlandı ağırlıklı

data = []
start_date = datetime(2023, 1, 1)

print("Veriler üretiliyor, lütfen bekleyin...")

for i in range(1, num_rows + 1):
    siparis_id = 1000 + i
    musteri_id = f"M-{random.randint(100, 999)}"
    kategori = random.choice(kategoriler)
    fiyat = random.randint(50, 15000)
    adet = random.randint(1, 5)
    
    # Rastgele bir tarih seçimi
    days_offset = random.randint(0, 300)
    siparis_tarihi = start_date + timedelta(days=days_offset)
    tarih_str = siparis_tarihi.strftime('%Y-%m-%d')
    
    sehir = random.choice(sehirler)
    durum = random.choice(durumlar)

    # --- KİRLİ VERİ OLUŞTURMA (Gerçek hayattaki hataları simüle ediyoruz) ---
    hata_ihtimali = random.random()

    if hata_ihtimali < 0.05:
        musteri_id = np.nan # %5 ihtimalle Müşteri ID eksik (NaN)
    elif 0.05 <= hata_ihtimali < 0.10:
        fiyat = -fiyat # %5 ihtimalle fiyat eksi (-) girilmiş
    elif 0.10 <= hata_ihtimali < 0.15:
        adet = np.nan # %5 ihtimalle adet bilgisi girilmemiş
    elif 0.15 <= hata_ihtimali < 0.20:
        tarih_str = siparis_tarihi.strftime('%d/%m/%Y') # %5 ihtimalle tarih formatı bozuk (GG/AA/YYYY)
    elif 0.20 <= hata_ihtimali < 0.25:
        sehir = sehir.lower() # %5 ihtimalle şehir isimleri küçük harfle yazılmış (örn: istanbul)

    data.append([siparis_id, musteri_id, kategori, fiyat, adet, tarih_str, sehir, durum])

# Veriyi Pandas DataFrame'e dönüştürme
df = pd.DataFrame(data, columns=['Siparis_ID', 'Musteri_ID', 'Urun_Kategorisi', 'Fiyat', 'Adet', 'Siparis_Tarihi', 'Sehir', 'Durum'])

# CSV dosyası olarak bilgisayara kaydetme 
df.to_csv('satis_verileri_kirli.csv', index=False, encoding='utf-8-sig')

print("Harika! 1000 satırlık kirli veri seti Türkçe karakter destekli olarak oluşturuldu!")
print(f"Toplam Eksik (Null) Değer Sayısı:\n{df.isnull().sum()}")

Veriler üretiliyor, lütfen bekleyin...
Harika! 1000 satırlık kirli veri seti Türkçe karakter destekli olarak oluşturuldu!
Toplam Eksik (Null) Değer Sayısı:
Siparis_ID          0
Musteri_ID         58
Urun_Kategorisi     0
Fiyat               0
Adet               50
Siparis_Tarihi      0
Sehir               0
Durum               0
dtype: int64


In [4]:
import pandas as pd

print("1. Kirli veri seti okunuyor...")
# Veriyi okuyoruz
df = pd.read_csv('satis_verileri_kirli.csv')

print("\n--- Temizleme Öncesi Hatalar ---")
print("Eksik (NULL) Veri Sayısı:\n", df.isnull().sum())
print("\nNegatif Fiyat İçeren Satır Sayısı:", len(df[df['Fiyat'] < 0]))

# ---------------------------------------------------------
# ADIM 1: Eksik (NULL) Verilerle Başa Çıkma
# ---------------------------------------------------------
# Senaryo A: Müşteri ID'si olmayan bir sipariş veritabanında tutarsızlık yaratır (Foreign Key hatası). 
# Bu yüzden Müşteri ID'si boş olan satırları tamamen siliyoruz. (dropna)
df.dropna(subset=['Musteri_ID'], inplace=True)

# Senaryo B: Siparişte 'Adet' girilmemişse, sistem hatası varsayıp varsayılan olarak 1 atıyoruz. (fillna)
df['Adet'] = df['Adet'].fillna(1)

# ---------------------------------------------------------
# ADIM 2: Mantıksız Verileri Düzeltme
# ---------------------------------------------------------
# Fiyat eksi (-) bir değer olamaz. Yanlışlıkla eksi girilenlerin mutlak değerini (abs) alıyoruz.
df['Fiyat'] = df['Fiyat'].abs()

# ---------------------------------------------------------
# ADIM 3: Metin / Karakter Standardizasyonu
# ---------------------------------------------------------
# 'istanbul' gibi tamamen küçük yazılmış şehir isimlerinin ilk harfini büyütüyoruz.
# (Türkçe i/I dönüşümünde Pandas hata yapmasın diye özel düzeltme de ekliyoruz)
df['Sehir'] = df['Sehir'].str.capitalize().replace({'Istanbul': 'İstanbul', 'Izmir': 'İzmir'})

# ---------------------------------------------------------
# ADIM 4: Tarih Formatı Standardizasyonu
# ---------------------------------------------------------
# Hem 'YYYY-MM-DD' hem 'DD/MM/YYYY' karışık girilmiş tarihleri, 
# veritabanının anlayacağı standart YYYY-MM-DD formatına çeviriyoruz.
df['Siparis_Tarihi'] = pd.to_datetime(df['Siparis_Tarihi'], format='mixed', dayfirst=True)

# ---------------------------------------------------------
# TEMİZLENMİŞ VERİYİ KAYDETME
# ---------------------------------------------------------
print("\n2. Veriler temizlendi! Yeni dosya kaydediliyor...")
df.to_csv('satis_verileri_temiz.csv', index=False, encoding='utf-8-sig')

print("\n--- Temizleme Sonrası Kontrol ---")
print("Eksik (NULL) Veri Sayısı:\n", df.isnull().sum())
print("Negatif Fiyat İçeren Satır Sayısı:", len(df[df['Fiyat'] < 0]))
print("\nİşlem Başarılı! 'satis_verileri_temiz.csv' dosyası oluşturuldu.")

1. Kirli veri seti okunuyor...

--- Temizleme Öncesi Hatalar ---
Eksik (NULL) Veri Sayısı:
 Siparis_ID          0
Musteri_ID         58
Urun_Kategorisi     0
Fiyat               0
Adet               50
Siparis_Tarihi      0
Sehir               0
Durum               0
dtype: int64

Negatif Fiyat İçeren Satır Sayısı: 58

2. Veriler temizlendi! Yeni dosya kaydediliyor...

--- Temizleme Sonrası Kontrol ---
Eksik (NULL) Veri Sayısı:
 Siparis_ID         0
Musteri_ID         0
Urun_Kategorisi    0
Fiyat              0
Adet               0
Siparis_Tarihi     0
Sehir              0
Durum              0
dtype: int64
Negatif Fiyat İçeren Satır Sayısı: 0

İşlem Başarılı! 'satis_verileri_temiz.csv' dosyası oluşturuldu.
